# Tabletop Games: Blackjack / 21 Probability Analysis

## Research Question
**What is the expected value (mean) and standard deviation of the total point value of a randomly dealt two-card starting hand in Blackjack (from a standard single 52-card deck), and how do these parameters describe the risk/volatility of going bust or hitting a strong hand?**

> *Note:* To answer this question, we define the probability distribution of starting hand values $X$, calculate its mean (expected value $\mu = E[X]$) and standard deviation ($\sigma = \sqrt{E[(X - \mu)^2]}$), and evaluate the likelihood of different hand outcomes.

### Step 1: Import Required Libraries

In [ ]:
import itertools
import pandas as pd
import numpy as np

### Step 2: Define Card Ranks, Suits, and Point Values in Blackjack

In [ ]:
# Standard 52-card deck components
suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']

# Point values in Blackjack (Number cards = face value, Face cards = 10, Ace = 11 initially)
card_values = {
    '2': 2, '3': 3, '4': 4, '5': 5, '6': 6,
    '7': 7, '8': 8, '9': 9, '10': 10,
    'J': 10, 'Q': 10, 'K': 10,
    'A': 11
}

### Step 3: Construct the Deck of Cards as a Pandas DataFrame

In [ ]:
# Build the 52-card deck
deck_data = []
for suit in suits:
    for rank in ranks:
        deck_data.append({
            'Rank': rank,
            'Suit': suit,
            'Card': f"{rank} of {suit}",
            'Value': card_values[rank]
        })

df_deck = pd.DataFrame(deck_data)
print(f"Total cards in deck: {len(df_deck)}")
df_deck.head()

### Step 4: Define Hand Value Calculation Logic for Blackjack

In [ ]:
def calculate_hand_value(card1_val, card2_val):
    """
    Calculates the best Blackjack value for a two-card starting hand.
    If both cards are Aces (11 + 11 = 22), one Ace counts as 1 so the hand total is 12.
    """
    total = card1_val + card2_val
    if total > 21 and (card1_val == 11 or card2_val == 11):
        total -= 10
    return total

### Step 5: Generate the Complete Sample Space of All Possible 2-Card Starting Hands

In [ ]:
# Generate all unique 2-card combinations without replacement from the 52-card deck: C(52, 2) = 1,326
all_hands = []
for card1, card2 in itertools.combinations(df_deck.itertuples(index=False), 2):
    hand_val = calculate_hand_value(card1.Value, card2.Value)
    all_hands.append({
        'Card 1': card1.Card,
        'Card 2': card2.Card,
        'Hand': f"{card1.Rank}-{card2.Rank}",
        'Hand Value': hand_val
    })

df_hands = pd.DataFrame(all_hands)
print(f"Total possible starting hands: {len(df_hands)}")
df_hands.head()

### Step 6: Derive the Probability Distribution of Hand Values

In [ ]:
# Count frequency of each hand total and compute discrete probabilities P(X = x)
prob_dist = df_hands['Hand Value'].value_counts().sort_index().reset_index()
prob_dist.columns = ['Hand Value (x)', 'Frequency']
prob_dist['P(X = x)'] = prob_dist['Frequency'] / len(df_hands)

# Add x * P(X = x) to help compute the expected value
prob_dist['x * P(x)'] = prob_dist['Hand Value (x)'] * prob_dist['P(X = x)']

prob_dist

### Step 7: Calculate Mean (Expected Value) and Standard Deviation

In [ ]:
# Expected Value (Mean): mu = sum(x * P(X = x))
mean_hand_value = prob_dist['x * P(x)'].sum()

# Variance: E[(X - mu)^2] = sum((x - mu)^2 * P(X = x))
prob_dist['(x - mu)^2 * P(x)'] = ((prob_dist['Hand Value (x)'] - mean_hand_value) ** 2) * prob_dist['P(X = x)']
variance_hand_value = prob_dist['(x - mu)^2 * P(x)'].sum()

# Standard Deviation: sigma = sqrt(variance)
std_hand_value = np.sqrt(variance_hand_value)

print("=== Blackjack 2-Card Starting Hand Distribution Metrics ===")
print(f"Mean (Expected Value, μ)   : {mean_hand_value:.4f}")
print(f"Variance (σ²)               : {variance_hand_value:.4f}")
print(f"Standard Deviation (σ)      : {std_hand_value:.4f}")
print(f"Natural Blackjack (21) Prob : {prob_dist.loc[prob_dist['Hand Value (x)'] == 21, 'P(X = x)'].values[0] * 100:.2f}%")

### Step 8: Import Visualization Libraries and Configure Plot Styling

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set Seaborn theme and figure parameters
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "sans-serif"

### Step 9: Plot the Probability Distribution with Matplotlib & Seaborn

In [ ]:
# Initialize figure and axis
plt.figure(figsize=(12, 6))

# Highlight the natural Blackjack (21) bar with a distinct accent color
palette = ['#e63946' if x == 21 else '#4575b4' for x in prob_dist['Hand Value (x)']]

# Create barplot of probabilities
ax = sns.barplot(
    data=prob_dist,
    x='Hand Value (x)',
    y='P(X = x)',
    palette=palette
)

# Add reference line for Expected Value (Mean)
# Map mean_hand_value to categorical x position
x_vals = list(prob_dist['Hand Value (x)'])
mean_idx = np.interp(mean_hand_value, x_vals, range(len(x_vals)))
plt.axvline(
    x=mean_idx,
    color='#d90429',
    linestyle='--',
    linewidth=2,
    label=f'Expected Value (μ = {mean_hand_value:.2f})'
)

# Titles and labels
plt.title('Probability Distribution of Blackjack Two-Card Starting Hand Values', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Hand Value (x)', fontsize=12, labelpad=10)
plt.ylabel('Probability P(X = x)', fontsize=12, labelpad=10)
plt.legend(frameon=True, facecolor='white', loc='upper left', fontsize=11)

### Step 10: Format Axes, Annotate Bar Values, and Display the Plot

In [ ]:
# Annotate each bar with its percentage
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f'{height * 100:.1f}%',
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom',
            fontsize=9,
            xytext=(0, 3),
            textcoords='offset points'
        )

# Set y-axis limit to leave room for labels
plt.ylim(0, prob_dist['P(X = x)'].max() * 1.15)
plt.tight_layout()
plt.show()